<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/Lab2aml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup and Preparation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda:0


## Data Augmentation and Dataloaders

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

if not os.path.exists('hymenoptera_data'):
    !wget https://download.pytorch.org/tutorial/hymenoptera_data.zip
    !unzip hymenoptera_data.zip
    !rm hymenoptera_data.zip

data_dir = 'hymenoptera_data'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=2) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes




In [ ]:
def train_model(model, criterion, optimizer, scheduler=None, num_epochs=3):
  for epoch in range(num_epochs):
    print(f'Epoch {epoch}/{num_epochs - 1}')
    print('-' * 10)

    for phase in ['train', 'val']:
      if phase == 'train':
        model.train()
      else:
        model.eval()

      running_loss = 0.0
      running_corrects = 0

      for inputs, labels in dataloaders[phase]:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        with torch.set_grad_enabled(phase == 'train'):
          outputs = model(inputs)
          _, preds = torch.max(outputs, 1)
          loss = criterion(outputs, labels)

          if phase == 'train':
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
      if phase == 'train' and scheduler is not None:
        scheduler.step()

      epoch_loss = running_loss / dataset_sizes[phase]
      epoch_acc = running_corrects.double() / dataset_sizes[phase]

      print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

  return model

## Design: CNN vs MLP

MLP Model

In [ ]:
class MLPModel(nn.Module):
  def __init__(self):
    super(MLPModel, self).__init__()
    self.flatten = nn.Flatten()
    self.fc = nn.Sequential(
        nn.Linear(224*224*3, 512),
        nn.ReLU(),
        nn.Linear(512, len(class_names))
    )

  def forward(self, x):
    x = self.flatten(x)
    return self.fc(x)

Custom CNN Model

In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN, self).__init__()
    self.features = nn.Sequential (
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )
    self.classifier = nn.Sequential(
        nn.Linear(32*56*56, 128),
        nn.ReLU(),
        nn.Linear(128, len(class_names))
    )

  def forward(self, x):
    x = self.features(x)
    x = x.view(x.size(0), -1)
    return self.classifier(x)


In [ ]:
criterion = nn.CrossEntropyLoss()

print("--- Training MLP Model ---")
model_mlp = MLPModel().to(device)
optimizer_mlp = optim.SGD(model_mlp.parameters(), lr=0.001, momentum=0.9)
train_model(model_mlp, criterion, optimizer_mlp, num_epochs=3);

print("--- Training Simple CNN Model ---")
model_cnn = SimpleCNN().to(device)
optimizer_cnn = optim.SGD(model_cnn.parameters(), lr=0.001, momentum=0.9)
train_model(model_cnn, criterion, optimizer_cnn, num_epochs=3);




--- Training MLP Model ---
Epoch 0/2
----------
train Loss: 4.3758 Acc: 0.5738
val Loss: 57.8949 Acc: 0.5163
Epoch 1/2
----------
train Loss: 23.5684 Acc: 0.5205
val Loss: 53.8132 Acc: 0.5033
Epoch 2/2
----------
train Loss: 61.8935 Acc: 0.4877
val Loss: 287.3468 Acc: 0.5163
--- Training Simple CNN Model ---
Epoch 0/2
----------
train Loss: 0.7126 Acc: 0.5041
val Loss: 0.6813 Acc: 0.5229
Epoch 1/2
----------
train Loss: 0.6733 Acc: 0.5779
val Loss: 0.6871 Acc: 0.5490
Epoch 2/2
----------
train Loss: 0.6682 Acc: 0.5861
val Loss: 0.6799 Acc: 0.6275


## Fine-tuning Pretrained Models

Method 1: Freezing Weights (ResNet18)

In [ ]:
print("\n--- Training ResNet18 ---")
model_resnet = models.resnet18(pretrained=True)
for param in model_resnet.parameters():
  param.requires_grad = False

num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Linear(num_ftrs, len(class_names))
model_resnet = model_resnet.to(device)

optimizer_res = optim.SGD(model_resnet.fc.parameters(), lr=0.001, momentum=0.9)
train_model(model_resnet, criterion, optimizer_res, num_epochs=3);



--- Training ResNet18 ---
Epoch 0/2
----------
train Loss: 0.6495 Acc: 0.6393
val Loss: 0.2952 Acc: 0.8693
Epoch 1/2
----------
train Loss: 0.5445 Acc: 0.7500
val Loss: 0.5452 Acc: 0.7843
Epoch 2/2
----------
train Loss: 0.6837 Acc: 0.6762
val Loss: 0.1987 Acc: 0.9477


Method 2: Reconstructing Layers (MobileNetV2)

In [ ]:
print("\n--- Training MobileNet V2 ")
model_mobilenet = models.mobilenet_v2(pretrained=True)
num_ftrs = model_mobilenet.classifier[1].in_features
model_mobilenet.classifier[1] = nn.Linear(num_ftrs, len(class_names))
model_mobilenet = model_mobilenet.to(device)

optimizer_mob = optim.SGD(model_mobilenet.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_mob, step_size=7, gamma=0.1)
train_model(model_mobilenet, criterion, optimizer_mob, scheduler=exp_lr_scheduler, num_epochs=3);


--- Training MobileNet V2 
Epoch 0/2
----------
train Loss: 0.6459 Acc: 0.6885
val Loss: 0.2868 Acc: 0.8824
Epoch 1/2
----------
train Loss: 0.5568 Acc: 0.7746
val Loss: 0.3176 Acc: 0.8627
Epoch 2/2
----------
train Loss: 0.4908 Acc: 0.7623
val Loss: 0.2881 Acc: 0.8758


## Training Loop and Results

In [ ]:
criterion = nn.CrossEntropyLoss()

print("--- Training ResNet18 (Method 1: Frozen Weights) ---")
optimizer_res = optim.SGD(model_resnet.fc.parameters(), lr=0.001, momentum=0.9)
model_resnet = train_model(model_resnet, criterion, optimizer_res)

print("--- Training MobileNet V2 (Method 2: Full Fine-tuning) ---")
optimizer_mob = optim.SGD(model_mobilenet.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_mob, step_size=7, gamma=0.1)
model_mobilenet = train_model(model_mobilenet, criterion, optimizer_mob)

--- Training ResNet18 (Method 1: Frozen Weights) ---
Epoch 0/2
----------
train Loss: 0.3660 Acc: 0.8443
val Loss: 0.2974 Acc: 0.8954
Epoch 1/2
----------
train Loss: 0.3357 Acc: 0.8525
val Loss: 0.2472 Acc: 0.9346
Epoch 2/2
----------
train Loss: 0.3909 Acc: 0.8238
val Loss: 0.2693 Acc: 0.9216
--- Training MobileNet V2 (Method 2: Full Fine-tuning) ---
Epoch 0/2
----------
train Loss: 0.6006 Acc: 0.7213
val Loss: 0.3116 Acc: 0.8954
Epoch 1/2
----------
train Loss: 0.6288 Acc: 0.7377
val Loss: 0.4124 Acc: 0.8235
Epoch 2/2
----------
train Loss: 0.6910 Acc: 0.7213
val Loss: 0.4869 Acc: 0.8105
